# Proyecto #2: Maze Solver

Este notebook implementa y compara diferentes algoritmos de búsqueda (BFS, DFS, Greedy, A*) para resolver laberintos representados en archivos de texto, siguiendo los requerimientos del Proyecto #2.

In [1]:
import time
import math
import heapq
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

class MazeSolver:
    def __init__(self, file_path):
        self.load_maze(file_path)
        # Jerarquía de movimientos: Arriba, Derecha, Abajo, Izquierda
        self.moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        
    def load_maze(self, file_path):
        with open(file_path, 'r') as f:
            self.grid = [list(line.strip()) for line in f.readlines()]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        self.start = self.find_value('2')
        self.end = self.find_value('3')
        
    def find_value(self, val):
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] == val:
                    return (r, c)
        return None

    def is_valid(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols and self.grid[r][c] != '1'

    def get_path(self, parent, current):
        path = []
        while current in parent:
            path.append(current)
            current = parent[current]
        path.append(self.start)
        return path[::-1]

    def manhattan(self, p1, p2):
        return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

    def euclidean(self, p1, p2):
        return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

    def calculate_branching_factor(self, nodes_explored, depth):
        if depth == 0 or nodes_explored <= 1:
            return 0
        return nodes_explored ** (1/depth)

    def bfs(self):
        start_time = time.time()
        queue = deque([self.start])
        parent = {}
        visited = {self.start}
        nodes_explored = 0
        
        while queue:
            curr = queue.popleft()
            nodes_explored += 1
            if curr == self.end:
                return self.get_path(parent, curr), nodes_explored, time.time() - start_time
            for dr, dc in self.moves:
                nr, nc = curr[0] + dr, curr[1] + dc
                if self.is_valid(nr, nc) and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    parent[(nr, nc)] = curr
                    queue.append((nr, nc))
        return None, nodes_explored, time.time() - start_time

    def dfs(self):
        start_time = time.time()
        stack = [self.start]
        parent = {}
        visited = {self.start}
        nodes_explored = 0
        
        while stack:
            curr = stack.pop()
            nodes_explored += 1
            if curr == self.end:
                return self.get_path(parent, curr), nodes_explored, time.time() - start_time
            for dr, dc in reversed(self.moves):
                nr, nc = curr[0] + dr, curr[1] + dc
                if self.is_valid(nr, nc) and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    parent[(nr, nc)] = curr
                    stack.append((nr, nc))
        return None, nodes_explored, time.time() - start_time

    def greedy(self, heuristic_type='manhattan'):
        start_time = time.time()
        h_func = self.manhattan if heuristic_type == 'manhattan' else self.euclidean
        pq = [(h_func(self.start, self.end), self.start)]
        parent = {}
        visited = {self.start}
        nodes_explored = 0
        
        while pq:
            _, curr = heapq.heappop(pq)
            nodes_explored += 1
            if curr == self.end:
                return self.get_path(parent, curr), nodes_explored, time.time() - start_time
            for dr, dc in self.moves:
                nr, nc = curr[0] + dr, curr[1] + dc
                if self.is_valid(nr, nc) and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    parent[(nr, nc)] = curr
                    heapq.heappush(pq, (h_func((nr, nc), self.end), (nr, nc)))
        return None, nodes_explored, time.time() - start_time

    def a_star(self, heuristic_type='manhattan'):
        start_time = time.time()
        h_func = self.manhattan if heuristic_type == 'manhattan' else self.euclidean
        pq = [(0 + h_func(self.start, self.end), 0, self.start)]
        parent = {}
        g_score = {self.start: 0}
        nodes_explored = 0
        
        while pq:
            f, g, curr = heapq.heappop(pq)
            nodes_explored += 1
            if curr == self.end:
                return self.get_path(parent, curr), nodes_explored, time.time() - start_time
            for dr, dc in self.moves:
                nr, nc = curr[0] + dr, curr[1] + dc
                if self.is_valid(nr, nc):
                    new_g = g + 1
                    if (nr, nc) not in g_score or new_g < g_score[(nr, nc)]:
                        g_score[(nr, nc)] = new_g
                        f = new_g + h_func((nr, nc), self.end)
                        parent[(nr, nc)] = curr
                        heapq.heappush(pq, (f, new_g, (nr, nc)))
        return None, nodes_explored, time.time() - start_time

    def visualize(self, path=None, title="Maze"):
        data = np.zeros((self.rows, self.cols))
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] == '1':
                    data[r, c] = 0
                elif self.grid[r][c] == '2':
                    data[r, c] = 0.5
                elif self.grid[r][c] == '3':
                    data[r, c] = 0.7
                else:
                    data[r, c] = 1
        
        plt.figure(figsize=(8, 8))
        plt.imshow(data, cmap='gray')
        if path:
            rs, cs = zip(*path)
            plt.plot(cs, rs, color='red', linewidth=1)
        plt.title(title)
        plt.axis('off')
        plt.show()

def run_all_algorithms(maze_file):
    solver = MazeSolver(maze_file)
    algos = [
        ("BFS", solver.bfs, {}),
        ("DFS", solver.dfs, {}),
        ("Greedy (Manhattan)", solver.greedy, {'heuristic_type': 'manhattan'}),
        ("Greedy (Euclidean)", solver.greedy, {'heuristic_type': 'euclidean'}),
        ("A* (Manhattan)", solver.a_star, {'heuristic_type': 'manhattan'}),
        ("A* (Euclidean)", solver.a_star, {'heuristic_type': 'euclidean'})
    ]
    
    print(f"{ 'Algoritmo':<20 } | { 'Longitud':<10 } | { 'Nodos':<10 } | { 'Tiempo (s)':<12 } | { 'Branching Factor':<15 }")
    print("-" * 75)
    
    for name, func, kwargs in algos:
        path, nodes, t = func(**kwargs)
        length = len(path) if path else 0
        b_factor = solver.calculate_branching_factor(nodes, length)
        print(f"{ name:<20 } | { length:<10 } | { nodes:<10 } | { t:<12.6f } | { b_factor:<15.4f }")
        solver.visualize(path, title=name)

def run_random_simulation(maze_file, num_trials=3):
    print(f"\nSimulación con {num_trials} puntos de partida aleatorios:\n")
    solver = MazeSolver(maze_file)
    free_cells = [(r, c) for r in range(solver.rows) for c in range(solver.cols) if solver.grid[r][c] == '0']
    trials = random.sample(free_cells, min(num_trials, len(free_cells)))
    
    for i, start_pos in enumerate(trials):
        print(f"Prueba {i+1}: Punto de partida {start_pos}")
        solver.start = start_pos
        path, nodes, t = solver.a_star('manhattan')
        length = len(path) if path else 0
        print(f"Resultado A* (Manhattan): Longitud={length}, Nodos={nodes}, Tiempo={t:.6f}s\n")

run_all_algorithms('test_maze.txt')
run_random_simulation('test_maze.txt', 3)

ValueError: Unknown format code '\x20' for object of type 'str'